In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-09 20:19:49--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  3.99MB/s    in 0.3s    

2026-04-09 20:19:50 (3.99 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [3]:
print(len(text))

1115394


In [4]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [5]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [6]:
stoi = { ch:i for i,ch in enumerate(chars) } # String to Integer
print(stoi)

{'\n': 0, ' ': 1, '!': 2, '$': 3, '&': 4, "'": 5, ',': 6, '-': 7, '.': 8, '3': 9, ':': 10, ';': 11, '?': 12, 'A': 13, 'B': 14, 'C': 15, 'D': 16, 'E': 17, 'F': 18, 'G': 19, 'H': 20, 'I': 21, 'J': 22, 'K': 23, 'L': 24, 'M': 25, 'N': 26, 'O': 27, 'P': 28, 'Q': 29, 'R': 30, 'S': 31, 'T': 32, 'U': 33, 'V': 34, 'W': 35, 'X': 36, 'Y': 37, 'Z': 38, 'a': 39, 'b': 40, 'c': 41, 'd': 42, 'e': 43, 'f': 44, 'g': 45, 'h': 46, 'i': 47, 'j': 48, 'k': 49, 'l': 50, 'm': 51, 'n': 52, 'o': 53, 'p': 54, 'q': 55, 'r': 56, 's': 57, 't': 58, 'u': 59, 'v': 60, 'w': 61, 'x': 62, 'y': 63, 'z': 64}


In [7]:
itos = { i:ch for i,ch in enumerate(chars) } # Integer to String
print(itos)

{0: '\n', 1: ' ', 2: '!', 3: '$', 4: '&', 5: "'", 6: ',', 7: '-', 8: '.', 9: '3', 10: ':', 11: ';', 12: '?', 13: 'A', 14: 'B', 15: 'C', 16: 'D', 17: 'E', 18: 'F', 19: 'G', 20: 'H', 21: 'I', 22: 'J', 23: 'K', 24: 'L', 25: 'M', 26: 'N', 27: 'O', 28: 'P', 29: 'Q', 30: 'R', 31: 'S', 32: 'T', 33: 'U', 34: 'V', 35: 'W', 36: 'X', 37: 'Y', 38: 'Z', 39: 'a', 40: 'b', 41: 'c', 42: 'd', 43: 'e', 44: 'f', 45: 'g', 46: 'h', 47: 'i', 48: 'j', 49: 'k', 50: 'l', 51: 'm', 52: 'n', 53: 'o', 54: 'p', 55: 'q', 56: 'r', 57: 's', 58: 't', 59: 'u', 60: 'v', 61: 'w', 62: 'x', 63: 'y', 64: 'z'}


In [8]:
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: [itos[i] for i in l]

In [9]:
encoded_text = encode(text[:1000])
print(encoded_text)

[18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 14, 43, 44, 53, 56, 43, 1, 61, 43, 1, 54, 56, 53, 41, 43, 43, 42, 1, 39, 52, 63, 1, 44, 59, 56, 58, 46, 43, 56, 6, 1, 46, 43, 39, 56, 1, 51, 43, 1, 57, 54, 43, 39, 49, 8, 0, 0, 13, 50, 50, 10, 0, 31, 54, 43, 39, 49, 6, 1, 57, 54, 43, 39, 49, 8, 0, 0, 18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 37, 53, 59, 1, 39, 56, 43, 1, 39, 50, 50, 1, 56, 43, 57, 53, 50, 60, 43, 42, 1, 56, 39, 58, 46, 43, 56, 1, 58, 53, 1, 42, 47, 43, 1, 58, 46, 39, 52, 1, 58, 53, 1, 44, 39, 51, 47, 57, 46, 12, 0, 0, 13, 50, 50, 10, 0, 30, 43, 57, 53, 50, 60, 43, 42, 8, 1, 56, 43, 57, 53, 50, 60, 43, 42, 8, 0, 0, 18, 47, 56, 57, 58, 1, 15, 47, 58, 47, 64, 43, 52, 10, 0, 18, 47, 56, 57, 58, 6, 1, 63, 53, 59, 1, 49, 52, 53, 61, 1, 15, 39, 47, 59, 57, 1, 25, 39, 56, 41, 47, 59, 57, 1, 47, 57, 1, 41, 46, 47, 43, 44, 1, 43, 52, 43, 51, 63, 1, 58, 53, 1, 58, 46, 43, 1, 54, 43, 53, 54, 50, 43, 8, 0, 0, 13, 50, 50, 10, 0, 35, 43, 1, 49, 52, 53, 61, 5, 

In [10]:
decoded_text = decode(encoded_text)
print(''.join(decoded_text))

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



**Đóng lô dữ liệu**

In [11]:
import torch

In [12]:
data = torch.tensor(encode(text), dtype=torch.long)
print(data[:1000])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57,  1, 47, 57,  1, 41, 

In [13]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(train_data)

tensor([18, 47, 56,  ..., 43, 56, 43])


In [14]:
block_size = 8
train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [15]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
print(x)
print(y)

tensor([18, 47, 56, 57, 58,  1, 15, 47])
tensor([47, 56, 57, 58,  1, 15, 47, 58])


In [16]:
for t in range(block_size):
  context = x[:t+1]
  target = y[t]
  print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [17]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
  data = train_data if split =='train' else val_data
  ix = torch.randint(len(data) - block_size, (batch_size,))
  x = torch.stack([data[i:i+block_size] for i in ix])
  y = torch.stack([data[i+1:i+block_size+1]for i in ix])
  return x,y

In [18]:
xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('target:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
target:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [23]:
for b in range(batch_size):
  for t in range(block_size):
    context = xb[b,: t+1]
    target = yb[b,t]
    print(f'when input is {context.tolist()} the target: {target}')

when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53, 56, 1, 58, 46] the target: 39
when input is [44, 53, 56, 1, 58, 46, 39] the target: 58
when input is [44, 53, 56, 1, 58, 46, 39, 58] the target: 1
when input is [52] the target: 58
when input is [52, 58] the target: 1
when input is [52, 58, 1] the target: 58
when input is [52, 58, 1, 58] the target: 46
when input is [52, 58, 1, 58, 46] the target: 39
when input is [52, 58, 1, 58, 46, 39] the t

In [166]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) # each token directly reads off the logits for the next token from a lookup table

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, _ = self(idx)
            # print(f'logits: {logits}')
            # print(f'logits shape: {logits.shape}')
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            # print(f'logits for the last time step: {logits.shape}')
            probs = F.softmax(logits, dim=-1) # (B, C)
            # print(f'probs: {probs}')
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx


m = BigramLanguageModel(vocab_size)
# print('-------')
# print(f'input batch: {xb}')
# print('-------')
# print(f'token embedding table: {m.token_embedding_table}') # each token directly reads off the logits for the next token from a lookup table
# print(f'token embedding table weights: {m.token_embedding_table.weight}') # the weights of the embedding table are the logits for the next token
# print(f'token embedding table shape: {m.token_embedding_table.weight.shape}') # the embedding table has vocab_size rows and vocab_size columns, because each token in the vocabulary can be represented as a one-hot vector of length vocab_size, and the embedding table maps each token to a vector of length vocab_size which represents the logits for the next token in the sequence.
# print('-------')
logits, _ = m(xb, yb)
print(f'logits shape: {logits.shape}')
print(logits)
# print(f'decoded logits: {decode(logits.argmax(dim=-1).tolist())}')
print('-------')
start_id = xb[0, 0].view(1, 1)
# start_context = xb[0:1, :4]
print(f'start_id: {start_id}')
print(f'start_id decoded: {decode(start_id[0].tolist())}')
# print(f'start_context: {start_context}')
# print(f'start_context decoded: {decode(start_context[0].tolist())}')
print(f'generated from start_id: {''.join(decode(m.generate(start_id, max_new_tokens=2)[0].tolist()))}')
# print(f'generated from start_context: {''.join(decode(m.generate(start_context, max_new_tokens=100)[0].tolist()))}')

logits shape: torch.Size([256, 65])
tensor([[ 1.1513,  1.0539,  3.4105,  ..., -0.5686,  0.9079, -0.1701],
        [ 1.0726,  0.7295, -0.6665,  ...,  0.3115, -1.7675,  0.6818],
        [-0.8109,  0.2410, -0.1139,  ...,  1.4509,  0.1836,  0.3064],
        ...,
        [ 1.6515, -0.0424, -0.7355,  ...,  0.8682,  2.0593, -0.8159],
        [-0.7470, -1.4852,  0.1714,  ...,  2.2019,  2.4498,  0.6347],
        [ 1.0726,  0.7295, -0.6665,  ...,  0.3115, -1.7675,  0.6818]],
       grad_fn=<ViewBackward0>)
-------
start_id: tensor([[39]])
start_id decoded: ['a']
generated from start_id: abL


In [101]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [204]:
batch_size = 32
for steps in range(100):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(f'loss: {loss.item()}')

loss: 4.650131702423096


In [205]:
print(''.join(decode(m.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist())))


GgiyuwzkpRSPkQczCR
x;AAviBogOj;ZZmfqf3mZ.yIVY'VJreviy-KTqmzaoQTbjxn!zdmNY MbM$gGFW:BS3C&vWKNJ&QtLhugqYQ? qHJOH$zb:eW:'AU?$ 
p3UN
xcORSPyWVkuv$zybBOFKtv'zkFXNCj
jlulvmOMMkVB-gd&q,& Mmy;A3cP CTW$GcUN&!:CqVVjKHI,Sq,SByNd,jxUAoiyZxQQJAc-,q.mrkz3fFEHeDXyaXesrgC'Kbev'lu$zd.sEBmJBPn!$'Q!ZVwNj:Cll;oC&vfqE-!?etRSKttWTX$QG:C UD$Rkq:Cp,v;?wRjgQWWxK-&o'MNX
LTT;w,i?.nn;m,Sl;AGJFi-tBFdbnxnzmi?gCaNVT$kzPclQ&QqHA-I-dgZRJ??VQbc;.?RRjux
xagTph$muCBOWnqk!Uuf&yaP c-3VRuRymV?:Xc'i:hitp$!uUN!B3lFMkGb
 ERj.?jlgUcoqiqp


In [206]:
# Clean imports for the demo section
import torch
import torch.nn as nn
from torch.nn import functional as F

# Fix randomness so the demo is reproducible
torch.manual_seed(1337)

print('Clean demo imports loaded.')

Clean demo imports loaded.


In [207]:
# Clean model definition
# This is the same bigram idea: each token directly maps to logits for the next token.
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Each token id looks up one row from a table of size vocab_size x vocab_size.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx: (B, T) token ids
        # logits: (B, T, C) where C = vocab_size
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            print(f'[forward] idx shape: {idx.shape}')
            print(f'[forward] logits before flatten: {logits.shape}')
            print(f'[forward] targets before flatten: {targets.shape}')
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            print(f'[forward] logits after flatten: {logits.shape}')
            print(f'[forward] targets after flatten: {targets.shape}')
            loss = F.cross_entropy(logits, targets)
            print(f'[forward] loss value: {loss.item():.6f}')

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx starts with shape (B, T_start).
        for step in range(max_new_tokens):
            print(f'[generate] step {step + 1}')
            print(f'[generate] current idx shape: {idx.shape}')
            print(f'[generate] current idx: {idx}')

            # Forward pass without targets: we only need logits for sampling.
            logits, _ = self(idx)
            print(f'[generate] logits from forward: {logits.shape}')

            # Only keep the last time step because we want the next token after the current context.
            logits = logits[:, -1, :]
            print(f'[generate] last-step logits: {logits.shape}')

            # Softmax converts raw scores into probabilities that sum to 1.
            probs = F.softmax(logits, dim=-1)
            print(f'[generate] probabilities: {probs}')

            # Sample the next token from the probability distribution.
            idx_next = torch.multinomial(probs, num_samples=1)
            print(f'[generate] sampled next token: {idx_next}')

            # Append the sampled token to the running sequence.
            idx = torch.cat((idx, idx_next), dim=1)
            print(f'[generate] updated idx: {idx}')
            print('----------------------------------------')

        return idx

print('Clean BigramLanguageModel is defined.')

Clean BigramLanguageModel is defined.


In [208]:
# Helper: one clean batch inspection with clear shapes
batch_size = 32
xb_clean, yb_clean = get_batch('train')

print('xb_clean shape:', xb_clean.shape)
print('yb_clean shape:', yb_clean.shape)
print('xb_clean[0]:', xb_clean[0])
print('yb_clean[0]:', yb_clean[0])

print('Decoded xb_clean[0]:', ''.join(decode(xb_clean[0].tolist())))
print('Decoded yb_clean[0]:', ''.join(decode(yb_clean[0].tolist())))

xb_clean shape: torch.Size([32, 8])
yb_clean shape: torch.Size([32, 8])
xb_clean[0]: tensor([24, 43, 58,  5, 57,  1, 46, 43])
yb_clean[0]: tensor([43, 58,  5, 57,  1, 46, 43, 39])
Decoded xb_clean[0]: Let's he
Decoded yb_clean[0]: et's hea


In [209]:
# Instantiate the clean model and inspect the parameter table
m_clean = BigramLanguageModel(vocab_size)

print('Model created.')
print('Embedding table shape:', m_clean.token_embedding_table.weight.shape)
print('First row of embedding table:', m_clean.token_embedding_table.weight[0])
print('Row 0 represents the logits vector for token id 0.')

Model created.
Embedding table shape: torch.Size([65, 65])
First row of embedding table: tensor([-0.6631, -0.2513,  1.0101,  0.1215,  0.1584,  1.1340, -1.1539, -0.2984,
        -0.5075, -0.9239,  0.5467, -1.4948, -1.2057,  0.5718, -0.5974, -0.6937,
         1.6455, -0.8030,  1.3514, -0.2759, -1.5108,  2.1048,  2.7630, -1.7465,
         1.4516, -1.5103,  0.8212, -0.2115,  0.7789,  1.5333,  1.6097, -0.4032,
        -0.8345,  0.5978, -0.0514, -0.0646, -0.4970,  0.4658, -0.2573, -1.0673,
         2.0089, -0.5370,  0.2228,  0.6971, -1.4267,  0.9059,  0.1446,  0.2280,
         2.4900, -1.2237,  1.0107,  0.5560, -1.5935, -1.2706,  0.6903, -0.1961,
         0.3449, -0.3419,  0.4759, -0.7663, -0.4190, -0.4370, -1.0012, -0.4094,
        -1.6669], grad_fn=<SelectBackward0>)
Row 0 represents the logits vector for token id 0.


In [210]:
# Forward-pass debug on one batch
logits_clean, loss_clean = m_clean(xb_clean, yb_clean)

print('After forward pass:')
print('logits_clean shape:', logits_clean.shape)
print('loss_clean:', loss_clean)
print('logits_clean sample row 0:', logits_clean[0])
print('Predicted token ids from argmax:', logits_clean.argmax(dim=-1)[:10])
print('Decoded argmax tokens:', ''.join(decode(logits_clean.argmax(dim=-1).tolist())[:10]))

[forward] idx shape: torch.Size([32, 8])
[forward] logits before flatten: torch.Size([32, 8, 65])
[forward] targets before flatten: torch.Size([32, 8])
[forward] logits after flatten: torch.Size([256, 65])
[forward] targets after flatten: torch.Size([256])
[forward] loss value: 4.692231
After forward pass:
logits_clean shape: torch.Size([256, 65])
loss_clean: tensor(4.6922, grad_fn=<NllLossBackward0>)
logits_clean sample row 0: tensor([ 1.8682,  0.7536, -0.1177, -0.1967, -0.9552, -0.8995, -0.9583, -0.5945,
         0.1321, -0.5406,  0.1405, -0.7321,  1.1796,  1.3316, -0.2094,  0.0960,
         0.9040, -0.4032,  0.3027, -0.8034, -1.2537, -1.5195,  0.7446,  1.1914,
        -0.8061, -0.6290,  1.2447, -2.4400,  0.8408, -0.3993, -0.6126, -0.6597,
         0.7624,  0.0691,  0.2990, -1.4717,  0.9950,  0.3608,  0.3161,  0.3504,
        -1.7823,  0.1339, -2.0973,  1.9108,  1.6555,  2.0254,  0.6044, -0.7006,
         0.8141,  0.2263, -0.8224, -1.1513,  0.1186, -0.3123, -0.6024, -0.1058,
        

In [211]:
# One clean generation example with a short start context
start_context_clean = xb_clean[0:1, :2]
print('start_context_clean shape:', start_context_clean.shape)
print('start_context_clean tensor:', start_context_clean)
print('start_context_clean decoded:', ''.join(decode(start_context_clean[0].tolist())))

generated_clean = m_clean.generate(start_context_clean, max_new_tokens=20)
print('generated_clean shape:', generated_clean.shape)
print('generated_clean tensor:', generated_clean)
print('generated_clean decoded:', ''.join(decode(generated_clean[0].tolist())))

start_context_clean shape: torch.Size([1, 2])
start_context_clean tensor: tensor([[24, 43]])
start_context_clean decoded: Le
[generate] step 1
[generate] current idx shape: torch.Size([1, 2])
[generate] current idx: tensor([[24, 43]])
[generate] logits from forward: torch.Size([1, 2, 65])
[generate] last-step logits: torch.Size([1, 65])
[generate] probabilities: tensor([[0.0040, 0.0074, 0.0032, 0.0061, 0.0043, 0.0267, 0.0339, 0.0228, 0.0046,
         0.0098, 0.0116, 0.0012, 0.0027, 0.0352, 0.0821, 0.0034, 0.0042, 0.0004,
         0.0126, 0.0065, 0.0074, 0.0035, 0.0503, 0.0024, 0.0067, 0.0138, 0.0251,
         0.0028, 0.1122, 0.0026, 0.0031, 0.0023, 0.0023, 0.0172, 0.0270, 0.0035,
         0.0076, 0.0224, 0.0271, 0.0033, 0.0039, 0.0083, 0.0090, 0.0161, 0.0016,
         0.0560, 0.0068, 0.0283, 0.0083, 0.0013, 0.0023, 0.0384, 0.0071, 0.0026,
         0.0253, 0.0012, 0.0008, 0.0030, 0.0140, 0.0623, 0.0106, 0.0014, 0.0036,
         0.0380, 0.0247]], grad_fn=<SoftmaxBackward0>)
[generate] sa

In [212]:
# Clean training loop with explicit debug output
optimizer_clean = torch.optim.AdamW(m_clean.parameters(), lr=1e-3)
batch_size = 32

for step in range(10):
    xb_clean, yb_clean = get_batch('train')
    logits_clean, loss_clean = m_clean(xb_clean, yb_clean)

    optimizer_clean.zero_grad(set_to_none=True)
    loss_clean.backward()
    optimizer_clean.step()

    print(f'[train] step={step + 1:02d} loss={loss_clean.item():.6f}')

print('Training demo finished.')

[forward] idx shape: torch.Size([32, 8])
[forward] logits before flatten: torch.Size([32, 8, 65])
[forward] targets before flatten: torch.Size([32, 8])
[forward] logits after flatten: torch.Size([256, 65])
[forward] targets after flatten: torch.Size([256])
[forward] loss value: 4.854258
[train] step=01 loss=4.854258
[forward] idx shape: torch.Size([32, 8])
[forward] logits before flatten: torch.Size([32, 8, 65])
[forward] targets before flatten: torch.Size([32, 8])
[forward] logits after flatten: torch.Size([256, 65])
[forward] targets after flatten: torch.Size([256])
[forward] loss value: 4.833645
[train] step=02 loss=4.833645
[forward] idx shape: torch.Size([32, 8])
[forward] logits before flatten: torch.Size([32, 8, 65])
[forward] targets before flatten: torch.Size([32, 8])
[forward] logits after flatten: torch.Size([256, 65])
[forward] targets after flatten: torch.Size([256])
[forward] loss value: 4.798682
[train] step=03 loss=4.798682
[forward] idx shape: torch.Size([32, 8])
[forw

In [213]:
# Final generation after the short clean training run
final_sample = m_clean.generate(idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=80)
print('Final sample shape:', final_sample.shape)
print('Final sample tensor:', final_sample)
print('Final sample decoded:')
print(''.join(decode(final_sample[0].tolist())))

[generate] step 1
[generate] current idx shape: torch.Size([1, 1])
[generate] current idx: tensor([[0]])
[generate] logits from forward: torch.Size([1, 1, 65])
[generate] last-step logits: torch.Size([1, 65])
[generate] probabilities: tensor([[0.0043, 0.0064, 0.0224, 0.0092, 0.0096, 0.0254, 0.0026, 0.0061, 0.0049,
         0.0032, 0.0141, 0.0018, 0.0024, 0.0147, 0.0045, 0.0041, 0.0425, 0.0037,
         0.0318, 0.0063, 0.0018, 0.0675, 0.1295, 0.0014, 0.0349, 0.0018, 0.0188,
         0.0067, 0.0179, 0.0379, 0.0409, 0.0055, 0.0036, 0.0149, 0.0078, 0.0078,
         0.0050, 0.0130, 0.0063, 0.0028, 0.0609, 0.0048, 0.0102, 0.0164, 0.0020,
         0.0202, 0.0094, 0.0103, 0.0986, 0.0024, 0.0225, 0.0143, 0.0017, 0.0023,
         0.0163, 0.0067, 0.0115, 0.0058, 0.0132, 0.0038, 0.0054, 0.0053, 0.0030,
         0.0054, 0.0015]], grad_fn=<SoftmaxBackward0>)
[generate] sampled next token: tensor([[40]])
[generate] updated idx: tensor([[ 0, 40]])
----------------------------------------
[generate] st

## The mathematical trick in self-attention

In [214]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])
